In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors



Given a dataset, match treated (﻿X equals 1﻿) to untreated (﻿X equals 0﻿) based on the confounder (﻿Z﻿).

Find the average treatment effect (each item corresponds to one counterfactual) where the counterfactual is the nearest item in the other group (you can use ﻿N e a r e s t N e i g h b o r s﻿ for this.)

Find the average treatment effect on the treated, where each treated item corresponds to a counterfactual untreated item, but we otherwise ignore the untreated items.

Find the average treatment effect on the untreated, where each untreated item corresponds to a counterfactual treated item, but we otherwise ignore the treated items.

Find the optimal treatment effect, which is the maximum treatment effect across all untreated items (i.e., it ends up considering only a single untreated item with its single counterfactual). 
Use the file homework_6.1.csv. 

Question 1


Which is closest to the average treatment effect? 

In [3]:

# Load the dataset
from pathlib import Path

# data/ lives at the repo root; walk up from the kernel's cwd (the notebook folder) to find it
csv_path = next(p / 'data' / 'homework_6.1.csv'
                for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'data' / 'homework_6.1.csv').exists())
df = pd.read_csv(csv_path)
df.head()

,Z,X,Y
0,0.548814,0,-0.823220
1,0.715189,1,0.842405
2,0.602763,1,0.898618
3,0.544883,0,-0.817325
4,0.423655,0,-0.635482


In [4]:
treated   = df[df['X'] == 1].reset_index(drop=True)
untreated = df[df['X'] == 0].reset_index(drop=True)

# Treated -> nearest untreated
nn_untreated = NearestNeighbors(n_neighbors=1).fit(untreated[['Z']])
_, idx_u = nn_untreated.kneighbors(treated[['Z']])
effect_treated = treated['Y'].values - untreated['Y'].values[idx_u.ravel()]

# Untreated -> nearest treated
nn_treated = NearestNeighbors(n_neighbors=1).fit(treated[['Z']])
_, idx_t = nn_treated.kneighbors(untreated[['Z']])
effect_untreated = treated['Y'].values[idx_t.ravel()] - untreated['Y'].values

# ATE: every unit (treated + untreated) contributes one effect
ate = np.concatenate([effect_treated, effect_untreated]).mean()
print("ATE:", ate)

ATE: 1.6952701427497885


## The core idea

With nearest-neighbor matching, every unit gets a counterfactual from the opposite group: the unit in that group whose `Z` is closest to its own.

- **Treated unit** (`X = 1`): effect = its own `Y` − `Y` of its nearest untreated neighbor.
- **Untreated unit** (`X = 0`): effect = `Y` of its nearest treated neighbor − its own `Y`.

Always compute treated minus untreated, so the sign means the same thing in both directions.

Question 1 (ATE)

First split the data and run matching in both directions. Questions 2–4 reuse these results.

Things to notice:
- `fit()` gets the group you're searching **in**; `kneighbors()` gets the group you're finding matches **for**.
- Use `[['Z']]` (double brackets): sklearn needs a 2-D input.
- `kneighbors` returns `(distances, indices)`. You only need the indices, and `.ravel()` turns them from shape `(n, 1)` into a flat array.


Question 2

Which is closest to the average treatment effect on the treated? 




In [5]:
att = effect_treated.mean()
print("ATT:", att)

ATT: 1.8464085071912946


Question 3


Which is closest to the average treatment effect on the untreated? 

In [6]:
atu = effect_untreated.mean()
print("ATU:", atu)

ATU: 1.5494765534751722


Question 4

Which is closest to the optimal treatment effect? 

In [7]:
optimal = effect_untreated.max()
print("Optimal treatment effect:", optimal)

Optimal treatment effect: 2.172469885577008


## What you should get

The data has 491 treated and 509 untreated rows.

| Quantity | Value | Averaged over |
|---|---|---|
| ATE | **≈ 1.70** | all 1000 units |
| ATT | **≈ 1.85** | 491 treated |
| ATU | **≈ 1.55** | 509 untreated |
| Optimal | **≈ 2.17** | max over untreated |

For each question, pick the option closest to these values.

**Sanity check:** ATT is larger than ATU, so the treatment helps units with higher `Z` (the treated group) more. ATE falls between the two because it's their average weighted by group size: (491 × 1.85 + 509 × 1.55) / 1000 ≈ 1.70. If your ATE doesn't fall between ATT and ATU, check the subtraction order.